# 04 — Random Walk Baseline

## Purpose

This notebook establishes the benchmark that every later forecasting model must beat.

For each currency and forecast horizon:

\[
\hat P_{t+h}^{RW} = P_t
\]

Currencies:
- USD
- GBP
- EUR
- JPY

Horizons:
- H1
- H3
- H7

Development evaluation years:
- 2019
- 2020
- 2021
- 2022
- 2023

## Core metrics

Only the main metrics are retained:

- RMSE
- MAE
- Forecast Skill vs Random Walk
- Theil's U
- Diebold–Mariano p-value

Because this notebook evaluates the Random Walk itself:

- Forecast Skill = 0
- Theil's U = 1
- DM p-value = not applicable

The final 2024–2025 holdout is **not used here**.

In [ ]:
# ============================================================
# 1. IMPORTS + CONFIGURATION
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

CURRENCIES = ["USD", "GBP", "EUR", "JPY"]
HORIZONS = [1, 3, 7]
VALIDATION_YEARS = [2019, 2020, 2021, 2022, 2023]

print("Random Walk benchmark configuration loaded.")

In [ ]:
# ============================================================
# 2. FIND PROJECT ROOT
# ============================================================

def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]

    for root in candidates:
        path = (
            root
            / "data"
            / "processed"
            / "02_development_model_dataset.csv"
        )

        if path.exists():
            return root.resolve()

    raise FileNotFoundError(
        "Could not find "
        "data/processed/02_development_model_dataset.csv. "
        "Run Script 02 first."
    )

PROJECT_ROOT = find_project_root()

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "02_development_model_dataset.csv"
)

TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
PRED_DIR = PROJECT_ROOT / "outputs" / "predictions"

for p in [TABLE_DIR, FIGURE_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path   :", DATA_PATH)

In [ ]:
# ============================================================
# 3. LOAD DEVELOPMENT DATA ONLY
# ============================================================

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["Date"]
)

for h in HORIZONS:
    target_date_col = f"Target_Date_H{h}"

    if target_date_col in df.columns:
        df[target_date_col] = pd.to_datetime(
            df[target_date_col],
            errors="coerce"
        )

df = (
    df
    .sort_values("Date")
    .reset_index(drop=True)
)

print("Shape:", df.shape)
print(
    "Date range:",
    df["Date"].min().date(),
    "to",
    df["Date"].max().date()
)

assert df["Date"].max() < pd.Timestamp("2024-01-01")

print("PASS — final holdout is not present.")

# 4. Metric functions

In [ ]:
# ============================================================
# 4. CORE METRICS
# ============================================================

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return float(
        np.sqrt(
            np.mean(
                (y_true - y_pred) ** 2
            )
        )
    )


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return float(
        np.mean(
            np.abs(
                y_true - y_pred
            )
        )
    )


def forecast_skill(model_rmse, rw_rmse):
    if rw_rmse == 0:
        return np.nan

    return float(
        1 - model_rmse / rw_rmse
    )


def theils_u(model_rmse, rw_rmse):
    if rw_rmse == 0:
        return np.nan

    return float(
        model_rmse / rw_rmse
    )

# 5. Optional Diebold–Mariano helper

Later model notebooks will use this function to compare a challenger directly against Random Walk.

For the Random Walk benchmark itself, DM is not calculated.

In [ ]:
# ============================================================
# 5. DIEBOLD-MARIANO TEST HELPER
# ============================================================

from scipy.stats import t as student_t


def diebold_mariano_test(
    y_true,
    pred_model,
    pred_rw,
    horizon=1
):
    y_true = np.asarray(y_true, dtype=float)
    pred_model = np.asarray(pred_model, dtype=float)
    pred_rw = np.asarray(pred_rw, dtype=float)

    mask = (
        np.isfinite(y_true)
        & np.isfinite(pred_model)
        & np.isfinite(pred_rw)
    )

    y_true = y_true[mask]
    pred_model = pred_model[mask]
    pred_rw = pred_rw[mask]

    n = len(y_true)

    if n < 20:
        return np.nan, np.nan

    loss_model = (
        y_true - pred_model
    ) ** 2

    loss_rw = (
        y_true - pred_rw
    ) ** 2

    # Positive d means the model has lower squared error than RW.
    d = loss_rw - loss_model

    d_bar = np.mean(d)

    lag = max(0, int(horizon) - 1)

    gamma0 = np.var(
        d,
        ddof=1
    )

    long_run_var = gamma0

    for k in range(1, lag + 1):
        cov_k = np.cov(
            d[k:],
            d[:-k],
            ddof=1
        )[0, 1]

        long_run_var += (
            2
            * (1 - k / (lag + 1))
            * cov_k
        )

    if (
        not np.isfinite(long_run_var)
        or long_run_var <= 0
    ):
        return np.nan, np.nan

    dm_stat = (
        d_bar
        / np.sqrt(
            long_run_var / n
        )
    )

    p_value = (
        2
        * student_t.sf(
            abs(dm_stat),
            df=n - 1
        )
    )

    return float(dm_stat), float(p_value)

# 6. Random Walk forecast construction

At every forecast origin:

\[
\hat P_{t+h}^{RW} = P_t
\]

So the current TT mid-rate is used directly as the forecast for H1, H3 and H7.

In [ ]:
# ============================================================
# 6. GENERATE RANDOM WALK PREDICTIONS
# ============================================================

prediction_frames = []

for cur in CURRENCIES:
    for h in HORIZONS:

        target_col = f"{cur}_target_H{h}"
        target_date_col = f"Target_Date_H{h}"

        temp = df[
            [
                "Date",
                target_date_col,
                f"{cur}_mid",
                target_col,
            ]
        ].copy()

        temp = temp.rename(
            columns={
                "Date": "Forecast_Origin",
                target_date_col: "Target_Date",
                f"{cur}_mid": "RW_Prediction",
                target_col: "Actual",
            }
        )

        temp["Currency"] = cur
        temp["Horizon"] = h

        temp = temp[
            temp["Actual"].notna()
            & temp["RW_Prediction"].notna()
            & temp["Target_Date"].notna()
        ].copy()

        temp["Validation_Year"] = (
            temp["Forecast_Origin"]
            .dt.year
        )

        temp["Error"] = (
            temp["Actual"]
            - temp["RW_Prediction"]
        )

        temp["Squared_Error"] = (
            temp["Error"] ** 2
        )

        temp["Absolute_Error"] = (
            temp["Error"].abs()
        )

        prediction_frames.append(temp)

rw_predictions = pd.concat(
    prediction_frames,
    ignore_index=True
)

print("Random Walk prediction rows:", len(rw_predictions))
display(rw_predictions.head())

# 7. Restrict evaluation to rolling development years

Only forecast origins in 2019–2023 are included in the main benchmark table.

In [ ]:
# ============================================================
# 7. FILTER DEVELOPMENT VALIDATION YEARS
# ============================================================

rw_eval = rw_predictions[
    rw_predictions["Validation_Year"]
    .isin(VALIDATION_YEARS)
].copy()

print(
    "Evaluation rows:",
    len(rw_eval)
)

print(
    "Evaluation years:",
    sorted(
        rw_eval["Validation_Year"]
        .unique()
        .tolist()
    )
)

# 8. Year-by-year Random Walk benchmark

In [ ]:
# ============================================================
# 8. YEARLY BENCHMARK TABLE
# ============================================================

yearly_rows = []

for (
    cur,
    h,
    year
), g in rw_eval.groupby(
    [
        "Currency",
        "Horizon",
        "Validation_Year"
    ]
):

    this_rmse = rmse(
        g["Actual"],
        g["RW_Prediction"]
    )

    this_mae = mae(
        g["Actual"],
        g["RW_Prediction"]
    )

    yearly_rows.append({
        "Currency": cur,
        "Horizon": int(h),
        "Validation_Year": int(year),
        "Observations": len(g),
        "RMSE": this_rmse,
        "MAE": this_mae,
        "Forecast_Skill_vs_RW": 0.0,
        "Theils_U": 1.0,
        "DM_pvalue_vs_RW": np.nan,
    })

rw_yearly = pd.DataFrame(
    yearly_rows
).sort_values(
    [
        "Currency",
        "Horizon",
        "Validation_Year"
    ]
)

display(rw_yearly)

rw_yearly.to_csv(
    TABLE_DIR / "04_random_walk_yearly_results.csv",
    index=False
)

# 9. Pooled 2019–2023 Random Walk benchmark

This is the main reference table later models will compare against.

In [ ]:
# ============================================================
# 9. POOLED BENCHMARK TABLE
# ============================================================

pooled_rows = []

for (
    cur,
    h
), g in rw_eval.groupby(
    [
        "Currency",
        "Horizon"
    ]
):

    this_rmse = rmse(
        g["Actual"],
        g["RW_Prediction"]
    )

    this_mae = mae(
        g["Actual"],
        g["RW_Prediction"]
    )

    pooled_rows.append({
        "Currency": cur,
        "Horizon": int(h),
        "Observations": len(g),
        "RMSE": this_rmse,
        "MAE": this_mae,
        "Forecast_Skill_vs_RW": 0.0,
        "Theils_U": 1.0,
        "DM_pvalue_vs_RW": np.nan,
    })

rw_pooled = pd.DataFrame(
    pooled_rows
).sort_values(
    [
        "Currency",
        "Horizon"
    ]
)

display(rw_pooled)

rw_pooled.to_csv(
    TABLE_DIR / "04_random_walk_pooled_results.csv",
    index=False
)

# 10. Benchmark by horizon

This gives a simple view of how the forecast problem becomes harder as the horizon increases.

In [ ]:
horizon_rows = []

for h, g in rw_eval.groupby("Horizon"):

    horizon_rows.append({
        "Horizon": int(h),
        "Observations": len(g),
        "RMSE": rmse(
            g["Actual"],
            g["RW_Prediction"]
        ),
        "MAE": mae(
            g["Actual"],
            g["RW_Prediction"]
        ),
    })

rw_by_horizon = pd.DataFrame(
    horizon_rows
).sort_values(
    "Horizon"
)

display(rw_by_horizon)

rw_by_horizon.to_csv(
    TABLE_DIR / "04_random_walk_by_horizon.csv",
    index=False
)

# 11. Benchmark stability by year

The chart below shows how Random Walk RMSE changes across validation years.

This is useful because a model should not beat RW in only one convenient year and fail badly elsewhere.

In [ ]:
for cur in CURRENCIES:

    subset = rw_yearly[
        rw_yearly["Currency"] == cur
    ]

    plt.figure(figsize=(10, 5))

    for h in HORIZONS:
        temp = subset[
            subset["Horizon"] == h
        ]

        plt.plot(
            temp["Validation_Year"],
            temp["RMSE"],
            marker="o",
            label=f"H{h}"
        )

    plt.title(
        f"{cur}/LKR Random Walk RMSE by Validation Year"
    )
    plt.xlabel("Validation Year")
    plt.ylabel("RMSE")
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR / f"04_{cur}_rw_rmse_by_year.png",
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()

# 12. Save prediction-level benchmark data

Later model notebooks will use the same Random Walk predictions for exact matched comparisons.

In [ ]:
# ============================================================
# 12. SAVE PREDICTIONS
# ============================================================

PREDICTION_PATH = (
    PRED_DIR
    / "04_random_walk_predictions_2019_2023.csv"
)

rw_eval.to_csv(
    PREDICTION_PATH,
    index=False
)

print("Saved:", PREDICTION_PATH)

# 13. Benchmark lookup table

This compact file is convenient for later scripts.

Each challenger will retrieve the matching Random Walk RMSE for the same currency and horizon.

In [ ]:
rw_lookup = (
    rw_pooled[
        [
            "Currency",
            "Horizon",
            "RMSE",
            "MAE",
        ]
    ]
    .rename(
        columns={
            "RMSE": "RW_RMSE",
            "MAE": "RW_MAE",
        }
    )
)

display(rw_lookup)

rw_lookup.to_csv(
    TABLE_DIR / "04_random_walk_lookup.csv",
    index=False
)

# 14. Sanity checks

In [ ]:
# ============================================================
# 14. SANITY CHECKS
# ============================================================

checks = []

expected_tasks = (
    len(CURRENCIES)
    * len(HORIZONS)
)

checks.append({
    "Check": "12 currency-horizon tasks",
    "Status": (
        "PASS"
        if len(rw_pooled) == expected_tasks
        else "FAIL"
    ),
    "Evidence": f"{len(rw_pooled)} tasks found"
})

checks.append({
    "Check": "Only 2019-2023 evaluation years",
    "Status": (
        "PASS"
        if set(
            rw_eval["Validation_Year"].unique()
        ).issubset(
            set(VALIDATION_YEARS)
        )
        else "FAIL"
    ),
    "Evidence": str(
        sorted(
            rw_eval["Validation_Year"]
            .unique()
            .tolist()
        )
    )
})

checks.append({
    "Check": "Random Walk skill fixed at zero",
    "Status": (
        "PASS"
        if (
            rw_pooled["Forecast_Skill_vs_RW"]
            .eq(0)
            .all()
        )
        else "FAIL"
    ),
    "Evidence": "Skill = 0 by benchmark definition"
})

checks.append({
    "Check": "Random Walk Theil U fixed at one",
    "Status": (
        "PASS"
        if (
            rw_pooled["Theils_U"]
            .eq(1)
            .all()
        )
        else "FAIL"
    ),
    "Evidence": "Theil U = 1 by benchmark definition"
})

quality_gate = pd.DataFrame(checks)

display(quality_gate)

if not quality_gate["Status"].eq("PASS").all():
    raise RuntimeError(
        "Random Walk benchmark sanity check failed."
    )

quality_gate.to_csv(
    TABLE_DIR / "04_quality_gate.csv",
    index=False
)

# 15. Final benchmark summary

The table below is the benchmark every future model must beat.

A challenger is better than Random Walk when:

\[
Forecast\ Skill > 0
\]

and equivalently:

\[
Theil's\ U < 1
\]

RMSE is the primary model-selection metric.

MAE is the secondary error metric.

DM p-value is used to determine whether the difference in forecast loss is statistically supported.

In [ ]:
print("=" * 76)
print("NOTEBOOK 04 — RANDOM WALK BASELINE COMPLETE")
print("=" * 76)

print(
    "Evaluation years:",
    VALIDATION_YEARS
)

print(
    "Currencies:",
    CURRENCIES
)

print(
    "Horizons:",
    HORIZONS
)

print("\nPOOLED RANDOM WALK BENCHMARK:")
display(rw_pooled)

print("\nBenchmark predictions saved to:")
print(PREDICTION_PATH)

print("\nIMPORTANT:")
print(
    "Every later model must use the same forecast origins "
    "and matched actual observations when compared with RW."
)

print(
    "Next notebook: 05_Statistical_Models.ipynb"
)